# Learning Fusion 0–2 — Hands-On: Frames, Calibration, Projection

This notebook lets you **see the numbers** behind sections **0, 1, 2** of `LEARNING_FUSION.md` / `LEARNING_FUSION.html`.
It is **self-contained** — no uploads needed — so it runs as-is in **Google Colab** (which already has numpy + matplotlib).

The math mirrors your own repo:
- `Calib` below  ↔  `fusion/common/sensors/calibration.py`
- projection helpers  ↔  `fusion/common/sensors/projection.py`

**Convention (KITTI):** camera frame — **x right, y down, z forward**. LiDAR (velo) frame — **x forward, y left, z up**.

> Run top-to-bottom once. Then change the point `p_velo` and re-run to build intuition. Open the matching section in `LEARNING_FUSION.html` side-by-side.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Calib: mirrors fusion/common/sensors/calibration.py (numpy methods only) ----
class Calib:
    """Holds P2 (3x4), R0_rect (3x3), Tr_velo_to_cam (3x4) as homogeneous 4x4 so
    composition is just matmul. Convention: cam x-right, y-down, z-forward."""
    def __init__(self, P2, R0_rect, Tr_velo_to_cam):
        self.P2 = np.asarray(P2, float).reshape(3, 4)
        self.R0 = np.eye(4); self.R0[:3, :3] = np.asarray(R0_rect, float).reshape(3, 3)
        self.V2C = np.eye(4); self.V2C[:3, :] = np.asarray(Tr_velo_to_cam, float).reshape(3, 4)
        self.P  = np.eye(4); self.P[:3, :] = self.P2

    def velo_to_cam(self, pts):
        pts = np.atleast_2d(np.asarray(pts, float)); n = pts.shape[0]
        h = np.hstack([pts[:, :3], np.ones((n, 1))])
        return (self.R0 @ self.V2C @ h.T).T[:, :3]

    def cam_to_image(self, pts_cam):
        pts_cam = np.atleast_2d(np.asarray(pts_cam, float)); n = pts_cam.shape[0]
        h = np.hstack([pts_cam, np.ones((n, 1))])
        img = (self.P @ h.T).T[:, :3]
        depth = img[:, 2]
        uv = img[:, :2] / np.where(depth == 0, 1e-9, depth)[:, None]
        return uv, depth

    def velo_to_image(self, pts):
        cam = self.velo_to_cam(pts); uv, depth = self.cam_to_image(cam)
        return uv, depth, cam

    def image_to_cam(self, uv, depth):
        uv = np.atleast_2d(np.asarray(uv, float)); depth = np.asarray(depth, float).reshape(-1)
        fx, fy = self.P2[0, 0], self.P2[1, 1]; cx, cy = self.P2[0, 2], self.P2[1, 2]
        x = (uv[:, 0] - cx) * depth / fx
        y = (uv[:, 1] - cy) * depth / fy
        return np.stack([x, y, depth], axis=1)

    def cam_to_velo(self, pts_cam):
        pts_cam = np.atleast_2d(np.asarray(pts_cam, float)); n = pts_cam.shape[0]
        h = np.hstack([pts_cam, np.ones((n, 1))])
        return (np.linalg.inv(self.R0 @ self.V2C) @ h.T).T[:, :3]

# ---- projection helpers: mirror common/sensors/projection.py (numpy) ----
def lidar_to_image(points, calib, H, W):
    uv, depth, _ = calib.velo_to_image(points[:, :3])
    valid = (depth > 0) & (uv[:, 0] >= 0) & (uv[:, 0] < W) & (uv[:, 1] >= 0) & (uv[:, 1] < H)
    return uv, depth, valid

def render_depth_image(points, calib, H, W, fill=0.0):
    uv, depth, valid = lidar_to_image(points, calib, H, W)
    img = np.full((H, W), fill, np.float32)
    uv, depth = uv[valid], depth[valid]
    if uv.shape[0] == 0: return img
    u, v = uv[:, 0].astype(int), uv[:, 1].astype(int)
    order = np.argsort(-depth)        # far first -> near overwrites (closest surface wins)
    u, v, depth = u[order], v[order], depth[order]
    img[v, u] = depth.astype(np.float32)
    return img

print("Calib + projection helpers ready (mirror common/sensors/calibration.py & projection.py)")


## Section 0 · Why fuse sensors at all?

A single sensor has hard failure modes:
- **Camera** — gives color/texture but **no distance**; goes blind in fog / night / glare.
- **LiDAR** — gives precise 3D distance but no color/texture; sparse far away.

Fusion's promise: each sensor covers the other's failure mode. The toy demo below shows it directly —
the camera sees *what*, LiDAR sees *how far*, and neither alone is enough.


In [ ]:
rng = np.random.default_rng(0)
H, W = 200, 600
objects = [(8, -3), (15, 0), (25, 4)]          # (depth_m, x_m) of three "cars"

cam_clear = np.full((H, W), 0.85, float)       # bright road, clear day
cam_fog   = np.full((H, W, 3), 0.5, float)     # grey washout = camera BLIND in fog
ld = np.full((H, W), 0.0, float)               # LiDAR depth strip (works in fog)

for depth, x in objects:
    u = W/2 + (x / depth) * 720                 # project x at given depth -> pixel (fx=720)
    cu = int(np.clip(u, 0, W-1)); cv = H // 2
    cam_clear[cv-15:cv+15, cu-15:cu+15] = 0.10  # "I see a dark blob"
    cam_fog[cv-15:cv+15, cu-15:cu+15] = 0.50    # invisible in fog
    ld[cv-3:cv+3, cu-2:cu+2] = depth            # LiDAR: exact depth even in fog

fig, ax = plt.subplots(2, 2, figsize=(12, 5))
ax[0,0].imshow(cam_clear, cmap='gray'); ax[0,0].set_title('Camera (clear): sees objects, NO depth'); ax[0,0].axis('off')
ax[0,1].imshow(cam_fog);        ax[0,1].set_title('Camera (fog): BLIND');                       ax[0,1].axis('off')
im = ax[1,0].imshow(ld, cmap='viridis'); ax[1,0].set_title('LiDAR (fog): exact depth, no texture'); ax[1,0].axis('off')
fig.colorbar(im, ax=ax[1,0], fraction=0.046, label='depth (m)')
ax[1,1].axis('off')
ax[1,1].text(0.05, 0.5,
    "Fusion promise:\n"
    "  camera  -> 'what'\n"
    "  lidar   -> 'how far'\n"
    "  each covers the other's\n"
    "  failure mode -> more\n"
    "  accurate AND more robust",
    transform=ax[1,1].transAxes, fontsize=12, va='center')
plt.tight_layout(); plt.show()


### 0·extra — Quantify it: a toy detection scenario

"Each sensor covers the other's failure" is only useful if you can *measure* it. Here is the toy
version of the benchmark's table: per scenario, how correct is each sensor, and does fusion beat both?


In [ ]:
# Toy scenarios: camera correctness vs LiDAR correctness (0..1) under each condition
scenarios = [
    ("clear day",        0.95, 0.90),
    ("night",            0.30, 0.88),   # camera blind, LiDAR fine
    ("fog",              0.25, 0.70),   # both degrade, LiDAR less
    ("heavy rain",       0.80, 0.40),   # LiDAR noisy, camera ok-ish
    ("glass storefront", 0.90, 0.20),   # LiDAR stray / sees through
]
print(f"{'scenario':18} {'cam':>6} {'lidar':>6} {'fused':>6}  winner")
print("-" * 54)
for name, c, l in scenarios:
    fused = min(max(c, l) + 0.05 * min(c, l), 1.0)   # best sensor + small agreement bonus
    winner = 'fused' if fused > max(c, l) else ('cam' if c >= l else 'lidar')
    print(f"{name:18} {c:6.2f} {l:6.2f} {fused:6.2f}  {winner}")
print("\n=> fused is never below the better sensor, and beats it whenever BOTH carry signal.")


### 0·extra — When fusion does NOT help (it has a real cost)

Fusion is not free. If the second sensor is pure noise, naïvely averaging it in makes you *worse*.
This is exactly why the project's blind-mode robustness sweep exists — to know when to *not* trust a sensor.


In [ ]:
rng = np.random.default_rng(3)
truth = 10.0
cam_est   = truth + rng.normal(0, 0.3, 5000)   # stereo camera depth: noisy but ok
lidar_bad = truth + rng.normal(0, 5.0, 5000)   # a FAULTY lidar: huge noise
fused_avg = 0.5 * (cam_est + lidar_bad)        # naive 50/50 fusion
def rmse(x): return float(np.sqrt(np.mean((x - truth) ** 2)))
print(f"RMSE  camera-only        : {rmse(cam_est):.3f} m")
print(f"RMSE  faulty-lidar-only  : {rmse(lidar_bad):.3f} m")
print(f"RMSE  naive 50/50 fuse   : {rmse(fused_avg):.3f} m   <-- WORSE than camera alone")
print("\nLesson: fusion pays off only when the second sensor carries information.")
print("Blind-mode sweeps measure exactly this -> so you fuse intelligently, not blindly.")


## Section 1 · Coordinate frames & calibration  →  `calibration.py`

KITTI uses three frames. Fusion = moving data between them:

```
velo (LiDAR)  --Tr_velo_to_cam-->  cam (unrectified)  --R0_rect-->  cam (rectified)  --P2-->  image (u,v,depth)
```

Three matrices, stored as homogeneous 4×4 so composition is just `@`:
- **`V2C`** = `Tr_velo_to_cam` (3×4) — LiDAR → camera, including a **translation** (LiDAR is mounted offset from the camera).
- **`R0`** = `R0_rect` (3×3) — stereo rectification (identity here, simplification).
- **`P`** = `P2` (3×4) — camera intrinsics (fx, fy, cx, cy).

Below we build a **realistic** calib: a rotation that maps velo (x-fwd, y-left, z-up) into cam (x-right, y-down, z-fwd), **plus a translation** so you can literally see the LiDAR→camera offset move the numbers.


In [ ]:
H, W = 375, 1242                       # KITTI image size
fx = fy = 720.0
cx, cy = W / 2., H / 2.                # 621.0, 187.5
P2 = np.array([[fx, 0, cx, 0],
               [0, fy, cy, 0],
               [0, 0,  1,  0]], float)

# Rotation: velo (x-fwd,y-left,z-up) -> cam (x-right,y-down,z-fwd)
#   cam_x = -velo_y ;  cam_y = -velo_z ;  cam_z = velo_x
R_velo2cam = np.array([[0, -1,  0],
                       [0,  0, -1],
                       [1,  0,  0]], float)

# Translation: LiDAR sits ~27 cm BEHIND and ~8 cm ABOVE the camera (in cam frame)
t = np.array([0.0, -0.08, -0.27])
Tr = np.hstack([R_velo2cam, t.reshape(3, 1)])     # 3x4
R0 = np.eye(3)                                    # rectification = identity (simplified)

calib = Calib(P2, R0, Tr)

np.set_printoptions(precision=4, suppress=True)
print("P2  (intrinsics, 3x4):\n", calib.P2); print()
print("R0_rect (3x3):\n", calib.R0[:3, :3]); print()
print("Tr_velo_to_cam (3x4)  -- note the translation in the last column:\n", calib.V2C[:3, :]); print()
print("chain:  velo --V2C--> cam(unrect) --R0--> cam(rect) --P2--> image")


In [ ]:
# One LiDAR point: a car 10 m ahead, 2 m to the LEFT, 0.5 m above the ground
p_velo = np.array([[10.0, 2.0, 0.5]])
print("point in VELO frame (x-fwd, y-left, z-up):", p_velo[0]); print()

# Step 1: velo -> cam(unrect) using V2C only (so you see the raw transform)
h = np.hstack([p_velo, np.ones((1, 1))])
cam_unrect = (calib.V2C @ h.T).T[:, :3]
print("after V2C  -> cam (unrect):", cam_unrect[0])

# Step 2: rectify with R0 (identity here, but the step matters for real KITTI)
cam_rect = (calib.R0 @ np.hstack([cam_unrect, np.ones((1, 1))]).T).T[:, :3]
print("after R0   -> cam (rect)  :", cam_rect[0], "   (unchanged: R0 = I)"); print()

print("READ THE TRANSLATION (the whole point of this section):")
print(f"  velo forward  x = 10.0  ->  cam z = {cam_rect[0,2]:.3f}   (forward distance; shifted by lidar offset -0.27)")
print(f"  velo left     y =  2.0  ->  cam x = {cam_rect[0,0]:.3f}   (LEFT is NEGATIVE cam-x: +x is right)")
print(f"  velo up       z =  0.5  ->  cam y = {cam_rect[0,1]:.3f}   (UP   is NEGATIVE cam-y: +y is down)")


### 1·extra — Decompose V2C into Rotation and Translation, applied separately

`velo_to_cam` does one matmul, but it is really two operations: **rotate**, then **translate**.
Splitting them shows you *why* the numbers move. We also check `R` is orthonormal (`R @ R.T = I`).


In [ ]:
R = calib.V2C[:3, :3]
t_vec = calib.V2C[:3, 3]
print("R (3x3 rotation):\n", R)
print("t (3, translation):", t_vec); print()
print("R @ R.T  (must be I if R is a clean rotation):\n", np.round(R @ R.T, 12)); print()

p = p_velo[0]
print("point velo                 :", p)
rotated = R @ p                       # step 1: pure rotation, no shift
print("after R only (no shift)    :", rotated)
cam_two_step = rotated + t_vec       # step 2: add the lidar->camera offset
print("after + t   (= cam unrect) :", cam_two_step)
print("matmul path cam_unrect     :", cam_unrect[0])
print("max diff between paths     :", np.abs(cam_two_step - cam_unrect[0]).max())


### 1·extra — Transform a BATCH of points (read the table)

Real fusion processes thousands of points at once. Here is a table so you can see how *each*
point's coordinates change as it moves velo → cam → pixel. Try editing `pts` and re-running.


In [ ]:
pts = np.array([
    [10.0,  2.0, 0.5],   # car ahead-left
    [10.0, -2.0, 0.5],   # car ahead-right
    [ 5.0,  0.0, 0.3],   # close object straight ahead
    [20.0,  0.0, 0.0],   # far ahead
    [ 3.0,  4.0,-1.0],   # low object, hard left, close
])
cam_pts = calib.velo_to_cam(pts)
uv_pts, d_pts = calib.cam_to_image(cam_pts)
print(f"{'velo (x,y,z)':24} {'cam (x,y,z)':24} {'pixel (u,v)':16} {'depth':>6} {'in-frame':>8}")
print("-" * 86)
for i in range(len(pts)):
    inf = bool(0 <= uv_pts[i, 0] < W and 0 <= uv_pts[i, 1] < H and d_pts[i] > 0)
    print(f"{str(pts[i].tolist()):24} {str(np.round(cam_pts[i], 3).tolist()):24} "
          f"{str(np.round(uv_pts[i], 1).tolist()):16} {d_pts[i]:6.2f} {str(inf):>8}")
print("\nObserve: forward distance (velo x) -> cam z; left (velo +y) -> cam -x; up (velo +z) -> cam -y.")


### 1·extra — The inverse, step by step: cam → velo

The transform is invertible (R is orthonormal). Inverting it is just: **subtract the translation, then
apply R⁻¹**. This is what `cam_to_velo` does, and it is how you turn a *camera-frame* detection back
into LiDAR/world coordinates.


In [ ]:
p_cam = cam_rect[0]
print("cam point                  :", p_cam)
R_inv = np.linalg.inv(R)
shifted = p_cam - t_vec               # undo the translation first
print("after - t                  :", shifted)
velo_rec = R_inv @ shifted            # then undo the rotation
print("after R^-1  (-> velo)      :", velo_rec)
print("original p_velo            :", p_velo[0])
print("reconstruction error       :", np.abs(velo_rec - p_velo[0]).max())


## Section 2 · Projection — 3D LiDAR → 2D pixels (and back)  →  `projection.py`

Projection = **matrix multiply, then divide by depth** (the *perspective divide*). The key insight:
**depth is not a separate measurement** — it falls out as the `Z'` coordinate after the intrinsics,
and it *survives* as the divisor. So one LiDAR point becomes `(u, v, depth)`.

We then render a whole cloud into a depth image (far-first so the **nearest** point wins),
and finally do the **back-projection** that lift-splat-shoot relies on: a pixel + a depth → a 3D point.


In [ ]:
# Forward: cam (3D) -> image (pixel + depth)
proj = (calib.P @ np.hstack([cam_rect, np.ones((1, 1))]).T).T[0, :3]   # (X', Y', Z')
uv, depth = calib.cam_to_image(cam_rect)

print("cam (rect)        :", cam_rect[0])
print("P2 @ [cam, 1]     :", proj, "    <- (X', Y', Z' = depth)")
print("perspective divide:  uv = (X', Y') / Z'   ->  uv =", uv[0])
print("depth = Z'        :", depth[0], "    <- depth SURVIVES projection (it IS the divisor)")
print(f"in-frame?  u in [0,{W}), v in [0,{H}):  {bool(0 <= uv[0,0] < W and 0 <= uv[0,1] < H)}")
print()
print(f"=> a car 10 m ahead, 2 m left, lands at pixel ({uv[0,0]:.1f}, {uv[0,1]:.1f}) "
      f"(center is ({cx:.0f},{cy:.0f}), so it's {cx-uv[0,0]:.0f}px left of center).")


In [ ]:
# Build a small synthetic scene and project it: wall + car + ground
rng = np.random.default_rng(1)
def box(cx, cy, cz, dx, dy, dz, n):
    return np.column_stack([rng.uniform(cx-dx/2, cx+dx/2, n),
                            rng.uniform(cy-dy/2, cy+dy/2, n),
                            rng.uniform(cz-dz/2, cz+dz/2, n)])

wall   = box(12, 0, 0,   0.2, 12, 3,   800)   # thin wall 12 m ahead, 12 m wide, 3 m tall
car    = box( 9, 2, 0.5, 1.0,  1, 1,   400)   # car 9 m ahead, 2 m left, 1 m tall
ground = box( 8, 0,-1.6,10,  14, 0.05, 600)   # ground ~1.6 m below the lidar
points = np.vstack([wall, car, ground])
points = np.hstack([points, np.zeros((len(points), 1))])   # append intensity col (0)

depth_img = render_depth_image(points, calib, H, W, fill=0)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
im = ax[0].imshow(depth_img, cmap='turbo')
ax[0].set_title('Projected LiDAR depth (m) — closest surface wins (far-first write)')
ax[0].set_xlabel('u (px)'); ax[0].set_ylabel('v (px)')
fig.colorbar(im, ax=ax[0], fraction=0.046, label='depth (m)')
ax[1].scatter(points[:, 0], points[:, 1], c=points[:, 2], s=5, cmap='viridis')
ax[1].set_title('BEV (top-down) of the same cloud')
ax[1].set_xlabel('velo x = forward (m)'); ax[1].set_ylabel('velo y = left (m)')
ax[1].set_xlim(0, 16); ax[1].set_ylim(-7, 7); ax[1].set_aspect('equal'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:
# Back-projection: (pixel, depth) -> 3D cam point  (the lift-splat-shoot atom)
uv_back    = np.array([[uv[0, 0], uv[0, 1]]])
depth_back = np.array([depth[0]])
cam_back   = calib.image_to_cam(uv_back, depth_back)

print("BACK-PROJECTION (image_to_cam):")
print("  original cam (rect) :", cam_rect[0])
print("  back-projected      :", cam_back[0], "   <- recovers the 3D cam point from (pixel, depth)"); print()

# Full round trip:  velo -> image -> cam -> velo   (should reconstruct the input)
rt_cam  = calib.image_to_cam(uv, depth)
rt_velo = calib.cam_to_velo(rt_cam)
print("ROUND TRIP (velo -> image -> velo):")
print("  original velo :", p_velo[0])
print("  round-trip    :", rt_velo[0])
print("  max error (m) :", np.abs(p_velo[0] - rt_velo[0]).max())
print()
print("Takeaway: projection is invertible as long as you KEEP depth. Lose depth and a")
print("pixel can never become a 3D point again — that is why lift-splat PREDICTS depth.")


### 2·extra — Perspective divide: raw `X', Y'` vs divided `u, v`

The divide by depth is the whole trick. Before it, the intrinsics produce huge numbers (`X', Y'` in
"image-plane units"); only after dividing by `Z'` do you get actual pixels. Depth is the divisor —
which is why it survives.


In [ ]:
h = np.hstack([cam_pts, np.ones((len(cam_pts), 1))])
proj = (calib.P @ h.T).T[:, :3]            # (X', Y', Z')
uv_div = proj[:, :2] / proj[:, 2:3]        # the perspective divide
print(f"{'depth Z\'':>9} {'X\' raw':>11} {'Y\' raw':>11} {'u = X\'/Z\'':>11} {'v = Y\'/Z\'':>11}")
print("-" * 58)
for i in range(len(cam_pts)):
    print(f"{proj[i,2]:9.3f} {proj[i,0]:11.2f} {proj[i,1]:11.2f} {uv_div[i,0]:11.2f} {uv_div[i,1]:11.2f}")
print("\nX', Y' are in the thousands; only /depth turns them into pixel coords.")


### 2·extra — Behind-the-camera & out-of-frame filtering (the `valid` mask)

Not every point lands in the image. The `valid` mask keeps only points that are (a) in front of the
camera (`depth > 0`) and (b) inside the pixel bounds. Points behind the camera are the subtle trap —
they project to *plausible-looking* pixels without the depth check.


In [ ]:
weird = np.array([
    [10,  0, 0.5],   # fine, in-frame
    [-5,  0, 0.5],   # BEHIND camera (velo x < 0) -> cam z negative
    [10, 30, 0.5],   # far left -> out of frame
    [10,  0, 60],    # very high -> likely out of frame vertically
])
uv2, d2, valid2 = lidar_to_image(np.hstack([weird, np.zeros((4, 1))]), calib, H, W)
print(f"{'velo':18} {'depth':>8} {'pixel':>16} {'valid':>6}  reason")
print("-" * 72)
for i in range(len(weird)):
    reason = 'OK' if valid2[i] else ('behind camera (depth<=0)' if d2[i] <= 0 else 'out of frame')
    print(f"{str(weird[i].tolist()):18} {d2[i]:8.2f} {str(np.round(uv2[i],1).tolist()):>16} "
          f"{str(bool(valid2[i])):>6}  {reason}")


### 2·extra — Why far-first sort: two points on the SAME ray

Multiple LiDAR points can project to the same pixel (they lie on the same ray from the camera, at
different depths). The renderer sorts **far-first** so the **nearest** point is written last and wins —
the closest surface is what is physically "in front". Here are two collinear points at 5 m and 9.73 m.


In [ ]:
# Build two points on the SAME ray (same pixel) at different depths, directly in cam frame
ray = np.array([[-0.2056, -0.0604, 1.0]])        # direction per unit z (from our 473,144 pixel)
camA = 5.00 * ray                                 # 5 m  along the ray (in front)
camB = 9.73 * ray                                 # 9.73 m along the ray (behind A)
cam_ray = np.vstack([camA, camB])
uv4, d4 = calib.cam_to_image(cam_ray)
print(f"point A  depth {d4[0]:.2f} m -> pixel {np.round(uv4[0],1).tolist()}")
print(f"point B  depth {d4[1]:.2f} m -> pixel {np.round(uv4[1],1).tolist()}   (same pixel!)")
order = np.argsort(-d4)                           # far first
labels = np.array(['A (5m, near)', 'B (9.73m, far)'])
print("write order (far first) :", [str(x) for x in labels[order]])
kept = d4[order[-1]]                              # last written = nearest
print(f"depth kept at that pixel: {kept:.2f} m  -> the NEARER surface wins (correct).")


### 2·extra — Back-project a grid of pixels: a pixel is a RAY (the lift-splat idea)

A pixel with no depth is not a point — it is a **ray** of infinitely many possible 3D points.
Back-projecting the same pixel at several depths shows the ray fanning out. Pinning one depth per pixel
turns rays into discrete 3D points: that is the "lift" in lift-splat-shoot.


In [ ]:
pixels = np.array([[621,144],[400,144],[800,144],[621,100],[621,250]])  # (u,v)
depths = [5, 10, 20, 40]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for (u, v) in pixels:
    pts = np.array([calib.image_to_cam(np.array([[u, v]]), np.array([d]))[0] for d in depths])
    ax[0].plot(pts[:, 2], pts[:, 1], '-o', label=f'pixel ({u},{v})')   # side view: z vs y
ax[0].set_title('Each pixel is a RAY (side view: z forward vs y down)')
ax[0].set_xlabel('cam z (forward, m)'); ax[0].set_ylabel('cam y (down, m)')
ax[0].axhline(0, color='gray', lw=.5); ax[0].grid(alpha=.3); ax[0].legend(fontsize=8)
pinned = np.array([calib.image_to_cam(np.array([[u, v]]), np.array([10.0]))[0] for (u, v) in pixels])
ax[1].scatter(pinned[:, 0], pinned[:, 2], c='red', s=40)              # top view: x vs z
ax[1].set_title('Pinning each ray at depth=10 m (top view: x vs z)')
ax[1].set_xlabel('cam x (right, m)'); ax[1].set_ylabel('cam z (forward, m)')
ax[1].set_aspect('equal'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print("Left : a pixel = a whole ray (infinite possible depths).")
print("Right: choose ONE depth per pixel -> discrete 3D points. That is the 'lift'.")
print("Lift-splat predicts a depth DISTRIBUTION, lifts each pixel to many weighted points,")
print("then 'splats' them into BEV -- so the camera branch can speak the same BEV language as LiDAR.")


### 2·extra — Focal length = zoom: change fx and watch the pixel (depth unchanged)

Intrinsics decide *where* a 3D point lands in the image, not *how far* it is. Doubling `fx` halves
the field of view and pushes pixels away from the principal point (a "zoom"). Depth is untouched.


In [ ]:
p = cam_rect[0:1]                              # the [-2, -0.58, 9.73] point
print(f"{'fx':>6} {'fy':>6} {'u':>9} {'v':>9} {'depth':>7}  note")
print("-" * 52)
for fx_test in [360, 720, 1440, 2880]:
    P2t = np.array([[fx_test, 0, cx, 0], [0, fx_test, cy, 0], [0, 0, 1, 0]], float)
    cb = Calib(P2t, np.eye(3), Tr)              # same extrinsics, only fx/fy change
    uvt, dt = cb.cam_to_image(p)
    note = 'wider FOV' if fx_test < 720 else ('baseline' if fx_test == 720 else 'narrower (zoom)')
    print(f"{fx_test:6d} {fx_test:6d} {uvt[0,0]:9.2f} {uvt[0,1]:9.2f} {dt[0]:7.2f}  {note}")
print("\nDoubling fx doubles the pixel's distance from the principal point -> zoom.")
print("depth is IDENTICAL across all rows: intrinsics never affect depth, only pixel placement.")


## Recap — what you just saw

| Step | Frame / space | What happened to the numbers |
|---|---|---|
| velo point | LiDAR (x-fwd, y-left, z-up) | `[10.0, 2.0, 0.5]` |
| → `V2C` | cam (x-right, y-down, z-fwd) | axes flip + **translation** from the LiDAR-mount offset |
| → `R0` | cam (rectified) | (identity here; real KITTI rectifies) |
| → `P2` + divide | image `(u, v)` + depth | perspective divide; **depth survives as `Z'`** |
| → `image_to_cam` | cam again | pixel + depth → 3D (invertible) |
| → `cam_to_velo` | velo again | round-trip error ≈ 0 |

**Now go test yourself** in `LEARNING_FUSION.html` → Self-Test **Q1** (frames + matrix chain), **Q2** (perspective divide), **Q7** (far-first sort). Re-take in 3 days.

Next notebooks in this series will cover **Section 3** (early/mid/late fusion, hands-on) and **Section 4** (robustness / blind-mode sweep).
